In [ ]:
# -*- coding: utf-8 -*-
import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

# Conjuntos fixos (NÃO MUDAR)
TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 50, 70}

RF_TEMP_PARAMS = dict(
    n_estimators=700, max_depth=22,
    min_samples_leaf=2, min_samples_split=4,
    max_features="sqrt", bootstrap=True,
    n_jobs=-1, random_state=7
)
# ÚNICO RF tabular por ponto (i,j)
RF_POINT_PARAMS = dict(
    n_estimators=1000, max_depth=24,
    min_samples_leaf=2, min_samples_split=6,
    max_features="sqrt", bootstrap=True,
    n_jobs=-1, random_state=42
)

USE_GAIN_EXTRAP = True   # ganho leve quando |ΔT_test| > ΔT_max_treino
WIN = 9                  # janela ímpar p/ média local e derivadas

# ===================== HELPERS =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

# ===== métricas =====
def rmsd(y_ref, y):   return float(np.sqrt(np.mean((y_ref - y)**2)))
def ccdm(y_ref, y):
    y1, y2 = y_ref - y_ref.mean(), y - y.mean()
    den = (np.linalg.norm(y1)*np.linalg.norm(y2))+1e-12
    rho = float(np.clip(np.dot(y1, y2)/den, -1, 1))
    return 1.0 - rho
def corr_per_sample(Y, Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(float(np.clip(num/den,-1,1)))
    return np.array(out)
def sam_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=float(np.clip(np.dot(y,yh)/den,-1,1))
        out.append(float(np.degrees(np.arccos(cosang))))
    return np.array(out)
def nrmse_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(float(rmse/(rng+1e-12)))
    return np.array(out)
def eval_all_metrics(Y_true, Y_pred):
    y1, y2 = Y_true.reshape(-1), Y_pred.reshape(-1)
    return dict(
        R2      = r2_score(y1, y2),
        RMSE    = float(np.sqrt(mean_squared_error(y1, y2))),
        MAE     = float(mean_absolute_error(y1, y2)),
        Corr    = float(corr_per_sample(Y_true, Y_pred).mean()),
        SAM_deg = float(sam_per_sample(Y_true, Y_pred).mean()),
        NRMSE   = float(nrmse_per_sample(Y_true, Y_pred).mean()),
        RMSD    = float(np.mean([rmsd(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
        CCDM    = float(np.mean([ccdm(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
    )
def print_metrics_block(title, m):
    print(f"\n== {title} ==")
    print(" | ".join([f"{k}={m[k]:.4f}" for k in ["R2","RMSE","MAE","Corr","SAM_deg","NRMSE","RMSD","CCDM"]]))

# ======== features locais por ponto (i,j) – CORRIGIDA ========
def local_feats_matrix(X, fhz, win=WIN):
    """
    Retorna matrizes (n, m) para:
      Z (original), loc_mean (média local centrada), slope (1ª derivada), curv (2ª derivada).
    """
    assert win % 2 == 1, "WIN deve ser ímpar"
    n, m = X.shape
    r = win // 2

    # média local centrada (shape n x m) — usando cumsum com zero à esquerda
    pad_left  = np.repeat(X[:, :1], r, axis=1)
    pad_right = np.repeat(X[:, -1:], r, axis=1)
    Xpad = np.hstack([pad_left, X, pad_right])        # (n, m+2r)

    csum = np.cumsum(Xpad, axis=1)
    csum = np.hstack([np.zeros((n,1), dtype=csum.dtype), csum])  # (n, m+2r+1)
    win_sum = csum[:, win:] - csum[:, :-win]                     # (n, m)
    loc_mean = win_sum / float(win)

    # derivadas centradas (em Hz)
    df = np.diff(fhz)                                  # (m-1,)
    df = np.concatenate([[df[0]], df, [df[-1]]])
    df = (df[:-1] + df[1:]) / 2.0                      # (m,)

    X_left  = np.hstack([X[:, :1], X[:, :-1]])
    X_right = np.hstack([X[:, 1:], X[:, -1:]])
    slope = (X_right - X_left) / (df[None, :] + 1e-12)

    X_ll = np.hstack([X[:, :1], X_left[:, 1:]])
    X_rr = np.hstack([X_right[:, :-1], X[:, -1:]])
    curv = (X_rr - 2*X + X_ll) / (df[None, :]**2 + 1e-12)

    return X, loc_mean, slope, curv  # todas (n,m)

def build_pointwise_dataset(X, T, fhz, y_ref, ref_temp, dt_from_rf=None, dt_max_train=None):
    """
    Constrói dataset tabular por ponto (i,j):
      Features: [ΔT, ΔT², ΔT³, Z_ij, mean_loc_ij, slope_ij, curv_ij, f̂_j, ĵ]
      Alvo: Δ_ij = y_ref[j] - X[i,j], padronizado por frequência.
    """
    n, m = X.shape
    Z, L, S, C = local_feats_matrix(X, fhz, win=WIN)

    # ΔT
    dT = (T - ref_temp).reshape(-1, 1) if dt_from_rf is None else dt_from_rf.reshape(-1,1)
    if dt_max_train is not None:
        dT = np.clip(dT, -dt_max_train, dt_max_train)
    dT2, dT3 = dT**2, dT**3

    # freq/índice normalizados
    fhat = (fhz - fhz.min())/(fhz.max() - fhz.min() + 1e-12)  # (m,)
    jhat = np.arange(m, dtype=float) / max(1.0, m-1)

    # repetir/tilar
    dT_rep  = np.repeat(dT,  m, axis=1)
    dT2_rep = np.repeat(dT2, m, axis=1)
    dT3_rep = np.repeat(dT3, m, axis=1)
    f_rep   = np.tile(fhat, (n,1))
    j_rep   = np.tile(jhat, (n,1))

    # alvo Δ e padronização por frequência
    Delta = (y_ref[None, :] - X)                 # (n,m)
    mu = Delta.mean(axis=0, keepdims=True)
    sd = Delta.std(axis=0, keepdims=True) + 1e-12
    Delta_std = (Delta - mu) / sd

    # flatten (n*m, p)
    X_tab = np.column_stack([
        dT_rep.reshape(-1,1), dT2_rep.reshape(-1,1), dT3_rep.reshape(-1,1),
        Z.reshape(-1,1), L.reshape(-1,1), S.reshape(-1,1), C.reshape(-1,1),
        f_rep.reshape(-1,1), j_rep.reshape(-1,1)
    ]).astype(float)
    y_tab = Delta_std.reshape(-1).astype(float)
    return X_tab, y_tab, mu, sd

# ===================== LOAD =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, _ = get_freq_columns(base_tr, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
freq_cols_te, _ = get_freq_columns(base_te, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order = np.argsort(fhz); common_cols = [common_cols[i] for i in order]; fhz = fhz[order]

X_tr_full = base_tr[common_cols].to_numpy(float)
X_te_full = base_te[common_cols].to_numpy(float)
T_tr_full = base_tr["temp_c"].to_numpy(float)
T_te_full = base_te["temp_c"].to_numpy(float)

# y_ref real @20 °C
pool_20=[]
if (base_tr["temp_c"]==REF_TEMP).any():
    pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
if (base_te["temp_c"]==REF_TEMP).any():
    pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
assert len(pool_20)>0, "Não há curva real @20°C!"
y_ref = np.median(np.vstack(pool_20), axis=0)

# ===================== RF-Temp (curva completa) =====================
X_all = np.vstack([X_tr_full, X_te_full])
T_all = np.concatenate([T_tr_full, T_te_full])
rf_temp = RandomForestRegressor(**RF_TEMP_PARAMS).fit(X_all, T_all)
print("\n== RF-Temp ==")
print(f"R²(all)={r2_score(T_all, rf_temp.predict(X_all)):.3f} | RMSE(all)={np.sqrt(mean_squared_error(T_all, rf_temp.predict(X_all))):.3f}")

# ===================== Conjuntos fixos =====================
tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()

X_tr = tr_restr[common_cols].to_numpy(float)
X_te = te_restr[common_cols].to_numpy(float)
T_tr = tr_restr["temp_c"].to_numpy(float)
T_te = te_restr["temp_c"].to_numpy(float)

# ===================== ΔT treino/teste =====================
DT_tr_raw       = (T_tr - REF_TEMP).reshape(-1,1)
DT_te_raw_pred  = (rf_temp.predict(X_te) - REF_TEMP).reshape(-1,1)  # ΔT previsto no teste

DT_max = float(np.max(np.abs(DT_tr_raw)))
print(f"[DEBUG] ΔT treino: min={DT_tr_raw.min():.1f}, max={DT_max:.1f}")
print(f"[DEBUG] ΔT teste (raw pred): min={DT_te_raw_pred.min():.1f}, max={DT_te_raw_pred.max():.1f}")

# ===================== Dataset tabular (treino) =====================
# treino usa ΔT real (sem clamp)
Xtr_tab, ytr_tab, mu_freq, sd_freq = build_pointwise_dataset(
    X_tr, T_tr, fhz, y_ref, REF_TEMP,
    dt_from_rf=DT_tr_raw, dt_max_train=None
)

# ===================== Treina ÚNICO RF tabular =====================
t0 = time.time()
rf_point = RandomForestRegressor(**RF_POINT_PARAMS)
rf_point.fit(Xtr_tab, ytr_tab)
print(f"[INFO] RF-point treino em {time.time()-t0:.1f}s | n amostras tabulares = {Xtr_tab.shape[0]:,}")

# ===================== Monta features do teste e prediz =====================
# clamp ΔT previsto às bordas vistas no treino (nas features)
DT_te_clamped = np.clip(DT_te_raw_pred, -DT_max, DT_max)

Xte_tab, _, _, _ = build_pointwise_dataset(
    X_te, T_te, fhz, y_ref, REF_TEMP,
    dt_from_rf=DT_te_clamped, dt_max_train=DT_max
)

t1 = time.time()
yte_hat_std = rf_point.predict(Xte_tab)
print(f"[INFO] RF-point predição em {time.time()-t1:.1f}s")

# Despadroniza por frequência e remonta curva compensada
n_te, m = X_te.shape
Yte_hat_std = yte_hat_std.reshape(n_te, m)
Yte_hat = Yte_hat_std * sd_freq + mu_freq
Y_te_hat = X_te + Yte_hat

# Extrapolação leve por ganho (opcional)
if USE_GAIN_EXTRAP:
    gamma_te = (np.abs(DT_te_raw_pred).reshape(-1,1) / (DT_max + 1e-12))
    gamma_te = np.maximum(1.0, gamma_te)
    Y_te_hat = X_te + (Yte_hat * gamma_te)

# ===================== MÉTRICAS =====================
Y_ref_te = np.tile(y_ref, (n_te,1))
def metrics_block(title, Yt, Yp):
    m = eval_all_metrics(Yt, Yp); print_metrics_block(title, m); return m

metrics_block("RF-Comp PROVA  (só RF)", Y_ref_te, Y_te_hat)

# ===== (Opcional) Park apenas para comparar =====
def park_va_for_shift(z_ref, z, k):
    n=len(z_ref)
    if k>=0: i0,i1,j0,j1=0,n-k,k,n
    else: i0,i1,j0,j1=-k,n,0,n+k
    if i1<=i0 or j1<=j0: return np.inf,0.0,None
    zr=z_ref[i0:i1]; zd=z[j0:j1]
    delta_s=(zr-zd).mean()
    diff=zr-(zd+delta_s)
    Va=np.sum(diff*diff)/(i1-i0)
    z_shift=np.full_like(z_ref,np.nan); z_shift[i0:i1]=zd+delta_s
    return Va,delta_s,z_shift

def park_compensate_curve(z_ref,z,max_shift=None):
    n=len(z_ref)
    if max_shift is None: max_shift=max(1,n//10)
    best=(np.inf,0.0,None)
    for k in range(-max_shift,max_shift+1):
        Va,ds,zc=park_va_for_shift(z_ref,z,k)
        if Va<best[0]: best=(Va,ds,zc)
    _,_,zc=best
    if np.isnan(zc).any():
        zc=zc.copy()
        idx_valid=np.where(~np.isnan(zc))[0]
        if len(idx_valid)>0:
            first,last=idx_valid[0], idx_valid[-1]
            zc[:first]=zc[first]; zc[last+1:]=zc[last]
        else: zc=z_ref.copy()
    return zc
def park_compensate_batch(y_ref,X):
    return np.vstack([park_compensate_curve(y_ref, X[i]) for i in range(X.shape[0])])

Y_te_hat_park = park_compensate_batch(y_ref, X_te)
metrics_block("Park PROVA (baseline)", Y_ref_te, Y_te_hat_park)

# ===================== PLOTS =====================
def plot_examples(fhz, y_ref, X_orig, Y_rf, Y_pk, base_te_df, T_pred, n=5):
    fhz_khz = fhz/1e3
    if len(base_te_df)==0:
        print("Sem amostras de teste para plot."); return
    idxs = np.random.choice(len(base_te_df), size=min(n, len(base_te_df)), replace=False)
    for i in idxs:
        T_real = float(base_te_df.iloc[i]["temp_c"])
        T_chute = float(T_pred[i])
        plt.figure(figsize=(8,4.5))
        plt.plot(fhz_khz, X_orig[i], label=f"Original @ {T_real:.0f}°C", lw=1.2)
        plt.plot(fhz_khz, y_ref,      label=f"Referência @ {REF_TEMP}°C", lw=1.8)
        plt.plot(fhz_khz, Y_rf[i],    label="RF-Comp (só RF)", lw=1.5)
        if Y_pk is not None:
            plt.plot(fhz_khz, Y_pk[i],label="Park (baseline)", lw=1.5)
        plt.xlabel("Frequência (kHz)"); plt.ylabel("Re{Z}")
        plt.title(f"Amostra {i} | T real={T_real:.0f}°C | RF-Temp previu {T_chute:.1f}°C")
        plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

print("\n== PLOTS DE EXEMPLOS ==")
T_pred_plot = rf_temp.predict(X_te)  # só para título
plot_examples(fhz, y_ref, X_te, Y_te_hat, Y_te_hat_park, te_restr, T_pred_plot, n=5)


In [ ]:
# ===== PLOTS INTERATIVOS: (1) ORIG+REF+PARK e (2) ORIG+REF+RF =====
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_OK = True
except Exception as _:
    PLOTLY_OK = False

fhz_khz = fhz/1e3
T_pred_plot = rf_temp.predict(X_te)

def _title(i):
    T_real = float(te_restr.iloc[i]["temp_c"])
    T_pred = float(T_pred_plot[i])
    return f"Amostra {i} | T real={T_real:.0f}°C | RF-Temp previu {T_pred:.1f}°C"

def plot_interactive_orig_ref_park():
    """
    Plot interativo com ORIG + REF + PARK (baseline).
    Se Plotly não estiver disponível, faz fallback para Matplotlib.
    """
    if PLOTLY_OK:
        i0 = 0
        fig = go.Figure()
        # traces base (visível = amostra i0)
        fig.add_trace(go.Scatter(x=fhz_khz, y=X_te[i0], name=f"Original @{te_restr.iloc[i0]['temp_c']:.0f}°C"))
        fig.add_trace(go.Scatter(x=fhz_khz, y=y_ref,      name=f"Referência @{REF_TEMP}°C"))
        fig.add_trace(go.Scatter(x=fhz_khz, y=Y_te_hat_park[i0], name="Park (baseline)"))

        # dropdown para trocar amostra
        buttons=[]
        for i in range(len(te_restr)):
            buttons.append(dict(
                label=f"#{i}  (T={float(te_restr.iloc[i]['temp_c']):.0f}°C)",
                method="update",
                args=[
                    {"y":[X_te[i], y_ref, Y_te_hat_park[i]]},
                    {"title": _title(i)}
                ]
            ))
        fig.update_layout(
            title=_title(i0),
            xaxis_title="Frequência (kHz)",
            yaxis_title="Re{Z}",
            updatemenus=[dict(type="dropdown", direction="down", buttons=buttons,
                              x=1.0, xanchor="right", y=1.15, yanchor="top")],
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
            margin=dict(l=50, r=20, t=80, b=50)
        )
        fig.show()
    else:
        # fallback matplotlib (com toolbar de zoom/pan da sua UI)
        import matplotlib.pyplot as plt
        i = np.random.randint(len(te_restr))
        plt.figure(figsize=(8,4.5))
        plt.plot(fhz_khz, X_te[i], label=f"Original @{te_restr.iloc[i]['temp_c']:.0f}°C")
        plt.plot(fhz_khz, y_ref,   label=f"Referência @{REF_TEMP}°C")
        plt.plot(fhz_khz, Y_te_hat_park[i], label="Park (baseline)")
        plt.title(_title(i)); plt.xlabel("Frequência (kHz)"); plt.ylabel("Re{Z}")
        plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()
        print("Plotly não disponível — exibido fallback Matplotlib. Para interatividade total, instale: pip install plotly")

def plot_interactive_orig_ref_rf():
    """
    Plot interativo com ORIG + REF + RF (compensada).
    Se Plotly não estiver disponível, faz fallback para Matplotlib.
    """
    if PLOTLY_OK:
        i0 = 0
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=fhz_khz, y=X_te[i0], name=f"Original @{te_restr.iloc[i0]['temp_c']:.0f}°C"))
        fig.add_trace(go.Scatter(x=fhz_khz, y=y_ref,      name=f"Referência @{REF_TEMP}°C"))
        fig.add_trace(go.Scatter(x=fhz_khz, y=Y_te_hat[i0], name="RF-Comp"))

        buttons=[]
        for i in range(len(te_restr)):
            buttons.append(dict(
                label=f"#{i}  (T={float(te_restr.iloc[i]['temp_c']):.0f}°C)",
                method="update",
                args=[
                    {"y":[X_te[i], y_ref, Y_te_hat[i]]},
                    {"title": _title(i)}
                ]
            ))
        fig.update_layout(
            title=_title(i0),
            xaxis_title="Frequência (kHz)",
            yaxis_title="Re{Z}",
            updatemenus=[dict(type="dropdown", direction="down", buttons=buttons,
                              x=1.0, xanchor="right", y=1.15, yanchor="top")],
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
            margin=dict(l=50, r=20, t=80, b=50)
        )
        fig.show()
    else:
        import matplotlib.pyplot as plt
        i = np.random.randint(len(te_restr))
        plt.figure(figsize=(8,4.5))
        plt.plot(fhz_khz, X_te[i], label=f"Original @{te_restr.iloc[i]['temp_c']:.0f}°C")
        plt.plot(fhz_khz, y_ref,   label=f"Referência @{REF_TEMP}°C")
        plt.plot(fhz_khz, Y_te_hat[i], label="RF-Comp")
        plt.title(_title(i)); plt.xlabel("Frequência (kHz)"); plt.ylabel("Re{Z}")
        plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()
        print("Plotly não disponível — exibido fallback Matplotlib. Para interatividade total, instale: pip install plotly")

# ---- Chame assim:
plot_interactive_orig_ref_park()
plot_interactive_orig_ref_rf()


In [ ]:
# ===== PLOTS APRESENTÁVEIS (fixos, não interativos) =====

def plot_apresentavel(i=0, save=False, prefix="compensacao"):
    """
    Gera duas figuras fixas (sem interatividade):
      1) Original + Referência + Park (baseline)
      2) Original + Referência + RF (compensada)
    Parâmetros:
      i: índice da amostra em te_restr
      save: se True, salva PNG em alta resolução
      prefix: prefixo do nome do arquivo salvo
    """
    import matplotlib.pyplot as plt

    # --- estilo simples e consistente ---
    plt.rcParams.update({
        "figure.figsize": (8.2, 4.6),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "lines.linewidth": 1.8,
    })

    fhz_khz = fhz / 1e3
    T_real = float(te_restr.iloc[i]["temp_c"])
    T_pred = float(rf_temp.predict(X_te[[i]])[0])

    # --------- FIGURA 1: ORIG + REF + PARK ----------
    fig1, ax1 = plt.subplots()
    ax1.plot(fhz_khz, X_te[i],              label=f"Original @ {T_real:.0f} °C",  color="#1f77b4")
    ax1.plot(fhz_khz, y_ref,                label=f"Referência @ {REF_TEMP} °C", color="#ff7f0e")
    ax1.plot(fhz_khz, Y_te_hat_park[i],     label="Park (baseline)",              color="#2ca02c")

    ax1.set_title(f"Amostra {i}  |  T real = {T_real:.0f} °C")
    ax1.set_xlabel("Frequência (kHz)")
    ax1.set_ylabel("Re{Z}")
    ax1.legend(loc="best", frameon=False)
    fig1.tight_layout()

    if save:
        fig1.savefig(f"{prefix}_orig_ref_park_i{i}.png", dpi=300)

    # --------- FIGURA 2: ORIG + REF + RF -------------
    fig2, ax2 = plt.subplots()
    ax2.plot(fhz_khz, X_te[i],              label=f"Original @ {T_real:.0f} °C",  color="#1f77b4")
    ax2.plot(fhz_khz, y_ref,                label=f"Referência @ {REF_TEMP} °C", color="#ff7f0e")
    ax2.plot(fhz_khz, Y_te_hat[i],          label="RF (compensada)",              color="#2ca02c")

    ax2.set_title(f"Amostra {i}  |  T real = {T_real:.0f} °C  |  T previsto = {T_pred:.1f} °C")
    ax2.set_xlabel("Frequência (kHz)")
    ax2.set_ylabel("Re{Z}")
    ax2.legend(loc="best", frameon=False)
    fig2.tight_layout()

    if save:
        fig2.savefig(f"{prefix}_orig_ref_rf_i{i}.png", dpi=300)

    plt.show()


plot_apresentavel(i=0, save=True)   # salva: compensacao_orig_ref_park_i0.png e compensacao_orig_ref_rf_i0.png
